# Load libraries

In [12]:
import json
from datetime import datetime, timezone
import pandas as pd

# Read needed file

In [3]:
path = "logs-mhs-wenyi12.json"

with open(path, "r", encoding="utf-8") as f:
    records = json.load(f)

type(records), len(records), records[0].keys()

(list,
 863,
 dict_keys(['_id', 'data', 'device', 'eventKey', 'eventType', 'game', 'playerId', 'sceneName', 'serverTimestamp', 'version']))

In [52]:
unique_event_types = sorted({
    r.get("eventType")
    for r in records
    if r.get("eventType") is not None
})

unique_event_types

['DialogueEvent',
 'PlayerPositionEvent',
 'Topographic Map Event',
 'TopographicMapEvent',
 'argumentationEvent',
 'argumentationNodeEvent',
 'argumentationToolEvent',
 'questEvent']

In [53]:
unique_scene_types = sorted({
    r.get("sceneName")
    for r in records
    if r.get("sceneName") is not None
})

unique_scene_types

['Unit 1 Dev', 'Unit 2 Prod (Refactor)']

# Test each progress point

In [5]:
pid = "wenyi12@mhs.mhs"

## U1.C1 Get your space legs

In [6]:
target_node = "DialogueNodeEvent:31:29"

has_31_29 = any(
    r.get("playerId") == pid and r.get("eventKey") == target_node
    for r in records
)

color = "green" if has_31_29 else None   
has_31_29, color

(True, 'green')

## U1.C2 Info and Intros

In [7]:
target_node = "DialogueNodeEvent:30:98"

has_30_98 = any(
    r.get("playerId") == pid and r.get("eventKey") == target_node
    for r in records
)

color = "green" if has_30_98 else None   
has_30_98, color

(True, 'green')

## U1.C3 Defend Expedition

In [9]:
quest_key = "questActiveEvent:34"

yellow_triggers = [
    "DialogueNodeEvent:70:25",
    "DialogueNodeEvent:70:33"
]

event_keys = {
    r.get("eventKey")
    for r in records
    if r.get("playerId") == pid and r.get("eventKey") is not None
}

has_quest_34 = quest_key in event_keys

color = None
if has_quest_34:
    has_yellow_trigger = any(k in event_keys for k in yellow_triggers)
    if has_yellow_trigger:
        color = "yellow"  
    else:
        color = "green"  

has_quest_34, has_yellow_trigger if has_quest_34 else None, color

(True, False, 'green')

## U1.C4: Unexpected turbulence

In [8]:
target_node = "DialogueNodeEvent:33:19"

has_33_19 = any(
    r.get("playerId") == pid and r.get("eventKey") == target_node
    for r in records
)

color = "green" if has_33_19 else None   
has_33_19, color

(True, 'green')

## U2.C1: Escape the ruins plus topographic glyph

In [4]:
yellow_nodes = ["DialogueNodeEvent:68:23", "DialogueNodeEvent:68:27", "DialogueNodeEvent:68:28", "DialogueNodeEvent:68:31"]
success_node = "DialogueNodeEvent:68:29"

event_keys = {
    r.get("eventKey")
    for r in records
    if r.get("playerId") == pid and r.get("eventKey") is not None
}

has_29 = success_node in event_keys
has_any_yellow = any(k in event_keys for k in yellow_nodes)

if has_29 and not has_any_yellow:
    color = "green"  # green
else:
    color = "yellow"

has_29, has_any_yellow, color

(True, False, 'green')

## U2.C2: Foraged forging plus finding toppo

In [40]:
START_KEY = "questFinishEvent:21"
END_KEY   = "DialogueNodeEvent:20:26"
GATE_KEY  = "DialogueNodeEvent:20:26"

TARGET_KEYS = [k.strip() for k in [
    "DialogueNodeEvent:18:99",  "DialogueNodeEvent:28:179", "DialogueNodeEvent:59:179",
    "DialogueNodeEvent:18:223", "DialogueNodeEvent:28:182", "DialogueNodeEvent:59:182",
    "DialogueNodeEvent:18:224", "DialogueNodeEvent:28:183", "DialogueNodeEvent:59:183"
]]

df = pd.json_normalize(records, sep=".")
df = df[df["playerId"] == pid].copy()

# Parse timestamps (use your actual column name; logs typically have serverTimestamp)
df["serverTimestamp"] = pd.to_datetime(df["serverTimestamp"])
df = df[df["serverTimestamp"].notna()].sort_values("serverTimestamp")

# Gate condition: only run this block if player has GATE_KEY
has_gate = (df["eventKey"] == GATE_KEY).any()

color = None
count_targets = None
start_ts = end_ts = None

if has_gate:
    # start_doc = earliest START_KEY
    start_rows = df[df["eventKey"] == START_KEY]
    if not start_rows.empty:
        start_ts = start_rows.iloc[0]["serverTimestamp"]

        # end_doc = earliest END_KEY after start_ts (>= start_ts)
        end_rows = df[(df["eventKey"] == END_KEY) & (df["serverTimestamp"] >= start_ts)]
        if not end_rows.empty:
            end_ts = end_rows.iloc[0]["serverTimestamp"]

            # count target keys within [start_ts, end_ts]
            window = df[(df["serverTimestamp"] >= start_ts) & (df["serverTimestamp"] <= end_ts)]
            count_targets = window["eventKey"].isin(TARGET_KEYS).sum()

            color = "green" if count_targets <= 1 else "yellow"
        else:
            color = None  # no end marker found after start
    else:
        color = None      # no start marker found

has_gate, start_ts, end_ts, count_targets, color

(np.True_,
 Timestamp('2026-02-05 23:33:42.544000+0000', tz='UTC'),
 Timestamp('2026-02-05 23:38:13.245000+0000', tz='UTC'),
 np.int64(0),
 'green')

## U2.C3: Finding Tera and Aryn

In [42]:
START_KEY = "DialogueNodeEvent:20:33"
END_KEY   = "DialogueNodeEvent:22:1"
GATE_KEY  = "DialogueNodeEvent:22:18"

TARGET_KEYS = set(k.strip() for k in [
    "DialogueNodeEvent:18:225", "DialogueNodeEvent:28:185", "DialogueNodeEvent:59:185",
    "DialogueNodeEvent:28:184", "DialogueNodeEvent:28:191", "DialogueNodeEvent:59:184", "DialogueNodeEvent:59:191",
    "DialogueNodeEvent:18:226", "DialogueNodeEvent:18:227", "DialogueNodeEvent:28:186", "DialogueNodeEvent:59:186",
    "DialogueNodeEvent:18:228", "DialogueNodeEvent:28:187", "DialogueNodeEvent:59:187",
    "DialogueNodeEvent:18:229", "DialogueNodeEvent:28:188", "DialogueNodeEvent:59:188",
    "DialogueNodeEvent:18:230", "DialogueNodeEvent:28:180", "DialogueNodeEvent:59:180",
    "DialogueNodeEvent:18:233", "DialogueNodeEvent:28:192", "DialogueNodeEvent:59:192",
    "DialogueNodeEvent:18:234", "DialogueNodeEvent:28:193", "DialogueNodeEvent:59:193",
    "DialogueNodeEvent:18:235", "DialogueNodeEvent:28:194", "DialogueNodeEvent:59:194",
    "DialogueNodeEvent:18:236", "DialogueNodeEvent:18:237", "DialogueNodeEvent:28:190", "DialogueNodeEvent:59:190"
])

def parse_ts(s):
    return datetime.fromisoformat(s.replace("Z", "+00:00"))

rows = []
for r in records:
    if r.get("playerId") != pid:
        continue
    ek = r.get("eventKey")
    ts = r.get("serverTimestamp")
    if ek is None or ts is None:
        continue
    try:
        rows.append((parse_ts(ts), ek))
    except Exception:
        pass

rows.sort(key=lambda x: x[0])

has_gate = any(ek == GATE_KEY for _, ek in rows)

color = None
count_targets = None
start_ts = end_ts = None

if has_gate:
    start_ts = next((ts for ts, ek in rows if ek == START_KEY), None)
    if start_ts is not None:
        end_ts = next((ts for ts, ek in rows if ek == END_KEY and ts >= start_ts), None)
        if end_ts is not None:
            count_targets = sum(1 for ts, ek in rows if start_ts <= ts <= end_ts and ek in TARGET_KEYS)
            color = "green" if count_targets <= 6 else "yellow"

has_gate, start_ts, end_ts, count_targets, color

(True,
 datetime.datetime(2026, 2, 5, 23, 39, 0, 464000, tzinfo=datetime.timezone.utc),
 datetime.datetime(2026, 2, 5, 23, 40, 46, 246000, tzinfo=datetime.timezone.utc),
 0,
 'green')

## U2.C4: Investigate the temple and watershed glyph

In [44]:
gate_key = "DialogueNodeEvent:23:17"
success_key = "DialogueNodeEvent:74:21"

bad_keys = [k.strip() for k in [
    " DialogueNodeEvent:74:16",
    " DialogueNodeEvent:74:17",
    " DialogueNodeEvent:74:20",
    " DialogueNodeEvent:74:22"
]]

event_keys = {
    r.get("eventKey")
    for r in records
    if r.get("playerId") == pid and r.get("eventKey") is not None
}

color = None
if gate_key in event_keys:
    has_74_21 = success_key in event_keys
    has_bad_feedback = any(k in event_keys for k in bad_keys)

    color = "green" if (has_74_21 and not has_bad_feedback) else "yellow"

(gate_key in event_keys), has_74_21 if gate_key in event_keys else None, has_bad_feedback if gate_key in event_keys else None, color

(True, True, False, 'green')

## U2.C5: Getting the band back together

In [47]:
GATE_KEY = "DialogueNodeEvent:23:42"

POS_KEYS = [
    "DialogueNodeEvent:26:165","DialogueNodeEvent:26:166","DialogueNodeEvent:26:167",
    "DialogueNodeEvent:26:168","DialogueNodeEvent:26:169","DialogueNodeEvent:26:170",
    "DialogueNodeEvent:26:171","DialogueNodeEvent:26:172","DialogueNodeEvent:26:173",
    "DialogueNodeEvent:26:174","DialogueNodeEvent:26:175","DialogueNodeEvent:26:176",
    "DialogueNodeEvent:26:177","DialogueNodeEvent:26:178","DialogueNodeEvent:26:179",
    "DialogueNodeEvent:26:180","DialogueNodeEvent:26:181","DialogueNodeEvent:26:182",
    "DialogueNodeEvent:26:183","DialogueNodeEvent:26:184","DialogueNodeEvent:26:185",
    "DialogueNodeEvent:26:186"
]

NEG_KEYS = [
    "DialogueNodeEvent:26:187","DialogueNodeEvent:26:188","DialogueNodeEvent:26:189",
    "DialogueNodeEvent:26:190","DialogueNodeEvent:26:191","DialogueNodeEvent:26:192",
    "DialogueNodeEvent:26:193","DialogueNodeEvent:26:194","DialogueNodeEvent:26:195",
    "DialogueNodeEvent:26:196","DialogueNodeEvent:26:197","DialogueNodeEvent:26:198",
    "DialogueNodeEvent:26:199","DialogueNodeEvent:26:200","DialogueNodeEvent:26:201",
    "DialogueNodeEvent:26:202","DialogueNodeEvent:26:203","DialogueNodeEvent:26:204",
    "DialogueNodeEvent:26:205","DialogueNodeEvent:26:206","DialogueNodeEvent:26:207",
    "DialogueNodeEvent:26:208","DialogueNodeEvent:26:209","DialogueNodeEvent:26:210",
    "DialogueNodeEvent:26:211"
]

player_event_keys = [
    r.get("eventKey")
    for r in records
    if r.get("playerId") == pid and r.get("eventKey") is not None
]
event_key_set = set(player_event_keys)

has_gate = GATE_KEY in event_key_set

color = None
pos_count = neg_count = None
score = None

if has_gate:
    pos_set = set(POS_KEYS)
    neg_set = set(NEG_KEYS)

    # Count occurrences (matches Mongo count_documents)
    pos_count = sum(1 for k in player_event_keys if k in pos_set)
    neg_count = sum(1 for k in player_event_keys if k in neg_set)

    score = pos_count - (neg_count / 3.0)
    color = "green" if score >= 4 else "yellow"  # 1=green, 2=yellow

has_gate, pos_count, neg_count, score, color

(True, 5, 0, 5.0, 'green')

## U2.C6: Drone tutorial plus data collection

In [49]:
GATE_KEY = "DialogueNodeEvent:20:35".strip()
PASS_KEY = "DialogueNodeEvent:20:43".strip()
YELLOW_KEYS = [k.strip() for k in ["DialogueNodeEvent:20:44", "DialogueNodeEvent:20:45"]]

event_keys = {
    r.get("eventKey")
    for r in records
    if r.get("playerId") == pid and r.get("eventKey") is not None
}

color = None

if GATE_KEY in event_keys:
    has_pass = PASS_KEY in event_keys
    if not has_pass:
        color = "yellow"  # yellow
    else:
        has_yellow_44_45 = any(k in event_keys for k in YELLOW_KEYS)
        color = "yellow" if has_yellow_44_45 else "green"  # 2=yellow, 1=green

(GATE_KEY in event_keys), (PASS_KEY in event_keys) if GATE_KEY in event_keys else None, color

(True, True, 'green')

## U2.C7: Watershed argument

In [51]:
GATE_KEY = "questFinishEvent:54".strip()
SUCCESS_KEY = "DialogueNodeEvent:27:7".strip()

NEG_KEYS = [k.strip() for k in [
    "DialogueNodeEvent:27:11", "DialogueNodeEvent:27:12", "DialogueNodeEvent:27:13", "DialogueNodeEvent:27:14",
    "DialogueNodeEvent:27:15", "DialogueNodeEvent:27:16", "DialogueNodeEvent:27:17", "DialogueNodeEvent:27:18",
    "DialogueNodeEvent:27:19", "DialogueNodeEvent:27:20", "DialogueNodeEvent:27:21", "DialogueNodeEvent:27:22",
    "DialogueNodeEvent:27:23", "DialogueNodeEvent:27:24", "DialogueNodeEvent:27:25", "DialogueNodeEvent:27:26",
    "DialogueNodeEvent:27:27", "DialogueNodeEvent:27:28", "DialogueNodeEvent:27:29", "DialogueNodeEvent:27:30"
]]

# Pull this player's eventKeys once (list keeps duplicates for counting)
player_event_keys = [
    r.get("eventKey")
    for r in records
    if r.get("playerId") == pid and r.get("eventKey") is not None
]
event_key_set = set(player_event_keys)

has_gate = GATE_KEY in event_key_set

color = None
has_success = None
neg_count = None

if has_gate:
    has_success = SUCCESS_KEY in event_key_set
    neg_set = set(NEG_KEYS)
    neg_count = sum(1 for k in player_event_keys if k in neg_set)  # occurrences, like count_documents

    color = "green" if (has_success and neg_count <= 3) else "yellow"  # 1=green, 2=yellow

has_gate, has_success, neg_count, color

(True, True, 0, 'green')